# Adjusting and comparing

In this notebook you will learn how to

- add a revision to a SUT
- replace cells with new values
- compare two versions of a SUT
- compare two versions of an inspection

**The revisions are invented.**

## Load the data

See [01_getting_started](01_getting_started.ipynb) for an explanation of these steps.

In [ ]:
import sys
from pathlib import Path

# Location of sutlab. Absolute, so that the notebook can be copied to another folder
sutlab_folder = Path(r"C:\Users\DstMove\Desktop\claude\projects\sutlab")
sys.path.insert(0, str(sutlab_folder))

import sutlab

data_folder = sutlab_folder / "docs" / "examples" / "data"
config_folder = sutlab_folder / "docs" / "examples" / "config"
years = [2016, 2017, 2018, 2019]

metadata = sutlab.load_metadata_from_excel(
    data_folder / "ta_columns.xlsx",
    data_folder / "ta_classifications.xlsx",
)

current_paths = [data_folder / f"ta_l_{year}.parquet" for year in years]
sut = sutlab.load_sut_from_separated_parquet(years, current_paths, metadata, "current_year")

## Add a revision

A revision is a small SUT that only holds the rows that should change. We load it from an Excel file in the folder `config`. The file has one row per cell that changes, and the same columns as the SUT. You can open it in Excel to see how it looks.

Our revision increases imports and household consumption of wine in 2019. Supply and use rows are in the same sheet, and are split when they are loaded.

In [ ]:
revision = sutlab.load_sut_from_combined_excel(
    config_folder / "revision_add.xlsx",
    metadata,
    "current_year",
)
revision.supply

Supply rows only have basic prices, so the other price columns are left empty for them in the file. Use rows have all the price columns.

In [ ]:
revision.use

`adjust_add_sut` adds the values to the cells that already exist. Cells that do not exist are created.

In [ ]:
sut_revised = sut.adjust_add_sut(revision)

`adjust_subtract_sut` does the opposite, so it takes us back to where we started.

In [ ]:
sut_restored = sut_revised.adjust_subtract_sut(revision)
sut_restored.inspect_sut_comparison(sut).summary

## Replace cells

Use `adjust_substitute_sut` when you know the new level rather than the change. The row is replaced completely, including all the price columns.

In [ ]:
new_level = sutlab.load_sut_from_combined_excel(
    config_folder / "revision_new_level.xlsx",
    metadata,
    "current_year",
)
new_level.use

In [ ]:
sut_revised = sut_revised.adjust_substitute_sut(new_level)

## Compare two SUTs

`inspect_sut_comparison` finds every cell that differs. Call it on the new version, and give it the old one.

In [ ]:
comparison = sut_revised.inspect_sut_comparison(sut)

The summary counts the changes in each table.

In [ ]:
comparison.summary

In [ ]:
comparison.supply

In [ ]:
comparison.use_purchasers

Small changes can be left out with `diff_tolerance`, and you can restrict the comparison to certain codes.

In [ ]:
large_changes = sut_revised.inspect_sut_comparison(
    sut,
    products="V22*",
    diff_tolerance=50000,
)
large_changes.use_purchasers

## Compare two inspections

Sometimes the effect on the results is more interesting than the individual cells. Every inspection can be compared with another one of the same kind. `diff` holds the differences and `rel` the relative changes.

In [ ]:
gdp_before = sut.inspect_gdp()
gdp_after = sut_revised.inspect_gdp()

gdp_comparison = gdp_after.inspect_tables_comparison(gdp_before)
gdp_comparison.diff.set_display_unit(1000).gdp

In [ ]:
gdp_comparison.rel.set_display_decimals(3).gdp

Look at the block `Balance`: the two sides of GDP no longer agree. Household consumption has risen by margins, taxes and VAT as well as the basic value. On the production side only taxes and VAT follow, because we did not raise the output of trade margins. The replaced cell adds to the difference too. A complete revision has to balance both.

---
Next: [06_aggregating](06_aggregating.ipynb)